# Track 08a — Evaluation (M1–M10)

### 평가(eval)란?

평가(eval)는 **에이전트가 "더 그럴듯해졌는지"가 아니라 "실제로 더 잘하는지"를 고정 테스트셋과 표준 지표로 확인하는 과정**입니다. eval 없이 데모만 늘리면 개선 여부를 감으로 판단하게 됩니다. 이 노트북에서는 표준 지표 **M1–M10**을 LLM 없이 **합성 trial(`TrialResult`)**로 직접 계산하면서, 각 지표가 *무엇을 왜* 재는지 확인합니다. 채점 기준은 `eval.metrics.*` 모듈이며, 이 노트북은 합성 입력을 만들어 그 모듈에 전달합니다.

### 이 노트북에서 보여줄 것

동일한 합성 trial 묶음을 10개 지표로 채점하고, 간단한 mini-check로 결과를 확인한 뒤(API 키 없이), API 키가 있으면 `eval.run`으로 실제 데이터셋 매트릭스까지 실행합니다.

| Session | 지표 | 무엇을 / 왜 |
|---|---|---|
| **1. 지도** | (개요) | M1–M10 ↔ `eval.metrics.*` 모듈 매핑과 등록된 datasets 10종. 도구 인자 정규화(canonical)도 먼저 봅니다. |
| **2. M1·M2** | Task Success · pass^k | 정답 일치(M1)와 *k회 모두 성공*해야 통과하는 신뢰도(M2, pass@k가 아니라 pass^k). |
| **3. M3·M4** | Tool Selection · Argument F1 | 올바른 도구 선택(이름 정규화 + Jaccard)과 인자 정확도(F1), 실패 원인까지. |
| **4. M5·M6** | Abstention · Schema | 도구가 필요 없을 때 호출을 참는지(M5)와 JSON 스키마 준수(M6, strict↔loose+AutoRepair). |
| **5. M7·M8·M9** | Efficiency · Redundancy · Faithfulness | 성공률 대비 토큰 비용, 중복 호출률, 근거 충실성. |
| **6. M10** | Empty-recovery | 빈 응답을 하네스가 재시도로 복구하는 비율. |
| **7. smoke** | 10개 지표 합산 | 모든 지표를 한 번에 계산하고 핵심 값을 assert로 검증. |
| **8. (선택) eval.run** | 실제 데이터셋 | API 키가 있으면 BFCL+IFEval 매트릭스 스모크 테스트. |

### 이 노트북을 마치면

- M1–M10 각 지표가 *무엇을 왜* 재는지 설명하고 합성 trial로 직접 계산할 수 있습니다.
- pass^k(≠pass@k), strict/loose 스키마 검증과 repair, abstention, faithfulness, recovery의 채점 방식을 구분할 수 있습니다.
- `metric_map.json`·`eval_table.md` 산출물을 만들고, API 키가 있으면 `eval.run`으로 실제 데이터셋에 연결할 수 있습니다.

> **한 줄 요약:** 합성 trial로 지표를 *직접 계산*해 보며, 단순한 점수가 아니라 *무엇을 재는지*를 익힙니다.

- **구성:** 각 `Session`은 가이드(텍스트) → 코드 → 출력 해석(텍스트) 순서로 구성되어 있습니다.
- **필요:** 합성 지표 계산은 **API 키 없이** 실행됩니다(로컬 `eval/` 패키지 필요). Session 8의 `eval.run`만 `EXAONE_API_KEY`/`EXAONE_BASE_URL`이 필요합니다(`HAS_API` 가드).
- **산출물:** `_out/metric_map.json`, `_out/eval_table.md`

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import logging
import warnings
# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력이 읽기 쉽도록 라이브러리 로그를 줄인다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치가 필요하다: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
# (en) Gate the optional live eval.run (Session 8) on API key presence; the synthetic
#      metric path below needs no LLM, so this notebook runs key-free up to Session 8.
# (kr) 선택 단계인 라이브 eval.run(Session 8)은 API 키가 있을 때만 실행한다. 아래 합성 지표 계산은
#      LLM이 필요 없으므로 Session 8 전까지는 API 키 없이 실행된다.
HAS_API = bool(os.environ.get("EXAONE_API_KEY", "").strip())
ROOT = exaone.project_root()
out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)

# (en) The eval.metrics modules are the scoring source of truth; we feed them synthetic trials.
# (kr) eval.metrics 모듈이 채점 기준이며, 여기서는 합성 trial을 입력한다.
from eval.metrics.types import ToolCallRecord, TrialResult


# (en) Tiny factory for synthetic trials — lets us compute every metric without an LLM.
# (kr) 합성 trial을 만드는 작은 factory. LLM 없이 모든 지표를 계산할 수 있게 한다.
def trial(task_id: str, **kw):
    defaults = dict(
        trial_id=f"t-{task_id}", task_id=task_id, dataset="track08.synthetic", runner="harness",
        final_content="", tool_calls=[], turns=1, input_tokens=50, output_tokens=30,
    )
    defaults.update(kw)
    return TrialResult(**defaults)


print("exaone", exaone.__version__, "| HAS_API =", HAS_API)

**출력 해석:** `exaone <버전> | HAS_API = True/False` 한 줄이 보이면 준비 완료입니다.

- `exaone` 버전과 `eval.metrics` import가 성공했다는 뜻입니다. 이후 합성 지표 셀은 **API 키가 없어도** 실행됩니다.
- `HAS_API = True`이면 Session 8의 라이브 `eval.run`까지 실행할 수 있고, `False`이면 Session 8만 `[SKIP]`됩니다. 나머지 셀은 그대로 실행됩니다.

## Session 1. M1–M10 지도

각 지표가 어떤 모듈에서 계산되는지 한눈에 살펴봅니다. 기준은 문자열 라벨이 아니라 `eval.metrics.*` 모듈입니다.


### Session 1-1. M1–M10 지표 매핑과 datasets 목록

**하는 일:** M1–M10 지표 매핑과 datasets 목록을 출력합니다.

**정상:** `M1`~`M10` 줄과 `datasets:`가 출력됩니다.

**의미:** 각 지표가 어떤 모듈에서 계산되는지 한눈에 확인합니다.


In [ ]:
from eval.datasets import available_datasets

METRIC_MAP = [
("M1", "Task Success Rate", "m1_task_success", "exact / judge"),
("M2", "pass^k Reliability", "m2_pass_k", "k trials; tau-bench pass4/pass8"),
("M3", "Tool Selection Accuracy", "m3_tool_selection", "strict + Jaccard"),
("M4", "Argument F1", "m4_argument_f1", "BFCL arg match"),
("M5", "Abstention Score", "m5_abstention", "expected_no_tools"),
("M6", "Schema Adherence", "m6_schema_adherence", "strict / loose; IFEval"),
("M7", "Token Efficiency Score", "m7_efficiency", "TSR / (mean_tokens/1000)"),
("M8", "Redundancy Rate", "m8_redundancy", "duplicate_calls / total_calls; lower is better"),
("M9", "Faithfulness Score", "m9_faithfulness", "judge + grounding; HaluBench"),
("M10", "Empty-response Recovery Score", "m10_empty_recovery", "metadata.recovery"),
]
for row in METRIC_MAP:
    print(f"{row[0]:3} {row[1]:32} module=eval.metrics.{row[2]}")
print("\ndatasets:", ", ".join(available_datasets()))


**출력 해석:** M1–M10 각 지표가 어떤 `eval.metrics.*` 모듈에서 계산되는지 표로 출력되고, 마지막에는 등록된 datasets 10종이 출력됩니다.

- `module=eval.metrics.mN_…` — 점수 계산의 **기준은 문자열 라벨이 아니라 이 모듈**입니다. 노트북은 입력을 만들고, 실제 채점은 모듈에 맡깁니다.
- `datasets:`에는 `bfcl_v3*`(5종)·`halubench`·`ifeval`·`tau_bench*`(3종) 총 10개가 보입니다. `tau_bench.retail`이 있으면 Session 7의 등록 검사가 통과합니다.

### Session 1-2. Canonical args (M3/M4/M8 공유)

**하는 일:** 도구 인자 비교의 두 축인 키 순서 정규화(`canonical_args`)와 값의 대소문자 비교(`values_match`)를 실행합니다.

**정상:** `canonical equal (key order): True`, `(case): False`, `values_match (case): True`

**의미:** 인자 채점에서는 **키 순서**를 정규화해 같은 것으로 보지만, **값의 대소문자**는 `canonical_args`가 구분하고 `values_match`가 별도로 처리합니다.

In [ ]:
from eval.metrics._canonical import canonical_args, values_match

# (en) canonical_args normalizes key ORDER only (sort_keys); it stays case-sensitive on values.
# (kr) canonical_args는 키 순서만 정규화(sort_keys)하며, 값의 대소문자는 구분한다.
order_a = {"city": "Seoul", "unit": "km"}
order_b = {"unit": "km", "city": "Seoul"}
print("canonical equal (key order):", canonical_args(order_a) == canonical_args(order_b))
print("canonical equal (case)     :", canonical_args({"city": "Seoul"}) == canonical_args({"city": "seoul"}))

# (en) values_match is the case-insensitive scalar comparison used inside M3/M4/M8.
# (kr) values_match는 M3/M4/M8 내부에서 쓰는 대소문자 무시 스칼라 비교다.
print("values_match (case)        :", values_match("Seoul", "seoul"))

**출력 해석:** 도구 인자 채점이 *무엇을 허용하고 무엇을 엄격하게 보는지*를 나누어 보여줍니다.

- `canonical equal (key order): True` — 키 순서만 다르면(`{city,unit}` vs `{unit,city}`) 같은 인자로 봅니다(`sort_keys` 정규화).
- `canonical equal (case): False` — 값의 대소문자(`Seoul` vs `seoul`)는 `canonical_args`가 **구분**합니다.
- `values_match (case): True` — 대소문자 무시는 `values_match`(M3/M4/M8 내부 비교)가 담당합니다.
- 핵심은 **키 순서에는 관대하지만, 값 정규화는 별도 단계(`values_match`)에서 처리한다는 점**입니다.

## Session 2. M1 Task Success · M2 pass^k

M1은 과업 성공 여부(정답 일치/judge)를 보고, M2는 *같은 과업을 k회 반복했을 때 모두 성공하는지*를 봅니다. 여기서 쓰는 값은 pass@k가 아니라 τ-bench 컨벤션의 pass^k입니다.


### Session 2-1. M1 Task Success · M2 pass^k

**하는 일:** 합성 trial로 M1 TSR(정답 일치)과 M2 pass^k(같은 과업 k회 *모두* 성공) 신뢰도를 계산합니다.

**정상:** `M1 TSR 0.333`, `M2 breakdown {pass_1:1.0, pass_2:1.0, pass_3:0.0}`

**의미:** M1은 과업 성공 여부를, M2는 *k회 모두 성공*하는 신뢰도를 봅니다. pass@k가 아니라 τ-bench 컨벤션의 pass^k입니다.

In [ ]:
from eval.metrics import m1_task_success, m2_pass_k
from eval.metrics.m1_task_success import TaskGold

golds = {
"t-a": TaskGold(task_id="t-a", answer={"status": "ok"}),
"t-b": TaskGold(task_id="t-b", answer="환불 14일"),
}
trials = [
trial("t-a", final_structured={"status": "ok"}),
trial("t-b", final_content="환불은 14일 이내입니다."),
trial("t-b", final_content="모릅니다.", runner="naive"),
]
m1 = m1_task_success.compute(trials, golds, mode="exact")
print("M1 TSR", m1.value, "n=", m1.n)

# (en) Same task run 3 times: harness passes twice, naive fails once -> pass^3 = 0.
# (kr) 같은 과업을 3회 실행한다. harness는 2회 성공하고 naive는 1회 실패하므로 pass^3 = 0이다.
by_task = {
"t-a": [
trial("t-a", trial_id="t-a-1", final_structured={"status": "ok"}),
trial("t-a", trial_id="t-a-2", final_structured={"status": "ok"}),
trial("t-a", trial_id="t-a-3", final_structured={"status": "fail"}, runner="naive"),
]
}
m2 = m2_pass_k.compute(by_task, golds, m1_task_success.score_trial_exact, ks=(1, 2, 3))
print("M2 breakdown:", m2.breakdown)


**출력 해석:** M1(성공률)과 M2(pass^k 신뢰도)를 봅니다.

- `M1 TSR 0.333 (n=3)` — trial 3건 중 1건만 정답과 일치합니다. exact 모드는 **정규화된 값이 정확히 같은지**를 보므로, "환불은 14일 이내입니다."는 정답 "환불 14일"과 다른 값으로 처리됩니다. 부분 문자열 포함만으로는 성공으로 보지 않습니다.
- `M2 breakdown {pass_1:1.0, pass_2:1.0, pass_3:0.0}` — 같은 과업 3회 중 naive가 1회 실패했습니다. pass^3은 *k회 모두 성공*해야 1이 되므로 한 번만 실패해도 0입니다. 이것이 **pass^k ≠ pass@k**의 핵심입니다.

## Session 3. M3 Tool Selection · M4 Argument F1

M3는 올바른 도구를 골랐는지(이름 정규화 후 strict + Jaccard)를 보고, M4는 인자를 정확히 채웠는지(F1)를 봅니다. `rag__retrieve`처럼 접두사가 붙은 이름도 정규화하면 `retrieve`와 매칭됩니다.


### Session 3-1. 핵심 코드

**하는 일:** 이 단계의 핵심 코드를 실행합니다.

**정상:** `M3`, `M4`, `failures`, `M3:`가 출력됩니다.

**의미:** M3는 올바른 도구를 골랐는지(이름 정규화 후 strict + Jaccard)를 보고, M4는 인자를 정확히 채웠는지(F1)를 봅니다. `rag__retrieve`처럼 접두사가 붙은 이름도 정규화하면 `retrieve`와 매칭됩니다.


In [ ]:
from eval.metrics import m3_tool_selection, m4_argument_f1

gold_calls = {
"tc1": [ToolCallRecord(name="retrieve", arguments={"q": "exaone agent"})],
"tc2": [ToolCallRecord(name="search", arguments={"q": "a"}), ToolCallRecord(name="fetch", arguments={"id": "1"})],
}
trials = [
trial("tc1", tool_calls=[ToolCallRecord(name="rag__retrieve", arguments={"q": "exaone agent"})]),
trial("tc2", tool_calls=[ToolCallRecord(name="search", arguments={"q": "a"}), ToolCallRecord(name="fetch", arguments={"id": "1"})]),
trial("tc2", trial_id="tc2-bad", runner="naive", tool_calls=[ToolCallRecord(name="search", arguments={"q": "wrong"})]),
]
m3 = m3_tool_selection.compute(trials, gold_calls)
m4 = m4_argument_f1.compute(trials, gold_calls)
print("M3", m3.value, "jaccard", m3.breakdown.get("jaccard"))
print("M4", m4.value)

# (en) Collect the strict-match failures so eval surfaces *why*, not just a score.
# (kr) strict 기준에서 어긋난 trial을 모아 점수뿐 아니라 *원인*까지 드러낸다.
failures = []
for t in trials:
    g = gold_calls.get(t.task_id)
    if g and m3_tool_selection.strict_match(
        [m3_tool_selection.normalize_tool_name(c.name) for c in t.tool_calls],
        [m3_tool_selection.normalize_tool_name(c.name) for c in g],
    ):
        continue
    failures.append({"trial_id": t.trial_id, "pred": [c.name for c in t.tool_calls]})
print("failures", failures)


**출력 해석:** M3(도구 선택)·M4(인자 F1)와 실패 원인을 봅니다.

- `M3 0.667 (jaccard 0.833)` — `rag__retrieve`는 정규화로 `retrieve`와 매칭됩니다. 다만 `tc2-bad`는 정답 도구 2개 중 `search`만 호출했기 때문에 strict 평균이 2/3입니다.
- `M4 0.667` — 인자 F1도 `tc2-bad`의 누락분만큼 낮아집니다.
- `failures [{trial_id: 'tc2-bad', pred: ['search']}]` — 점수뿐 아니라 **어느 trial이 왜 틀렸는지**까지 남겨, eval을 디버깅에 활용할 수 있게 합니다.

## Session 4. M5 Abstention · M6 Schema Adherence

M5는 도구가 필요 없을 때 호출을 참는지 보는 지표이고, M6는 JSON 스키마 준수를 보는 지표입니다. M6는 strict(`JsonExtractor`만 사용)와 loose(`AutoRepair`로 잘린 JSON까지 복구)로 나누어 보며, 둘의 차이가 `repair_gain`입니다.

### Session 4-1. 핵심 코드

**하는 일:** M5(절제)와 M6(strict/loose+repair)를 합성 trial로 계산합니다.

**정상:** `M5 0.5`, `M6 strict 0.333 → loose 0.667`, `repair_gain 0.333`

**의미:** M6의 loose 모드는 AutoRepair로 잘린 JSON을 복구해 strict보다 더 많은 케이스를 통과시킵니다. 그 차이가 하네스 repair 단계의 기여분입니다.

In [ ]:
from eval.metrics import m5_abstention, m6_schema_adherence
from eval.metrics.m6_schema_adherence import SchemaSpec

no_tools = {"irr1": True, "irr2": True}
trials_m5 = [
    trial("irr1", tool_calls=[]),
    trial("irr2", tool_calls=[ToolCallRecord(name="weather", arguments={"city": "Seoul"})], runner="naive"),
]
m5 = m5_abstention.compute(trials_m5, no_tools)

# (en) js2 is truncated mid-array: JsonExtractor (strict) fails, AutoRepair (loose) recovers
#      the full object with both keys -> this is exactly where repair_gain shows up.
# (kr) js2는 배열 중간에서 끊긴 JSON이다. JsonExtractor(strict)는 실패하고 AutoRepair(loose)가
#      두 키를 가진 객체를 복구한다. 이 차이에서 repair_gain이 드러난다.
trials_m6 = [
    trial("js1", final_content='{"title": "회의", "action_items": []}'),
    trial("js2", final_content='{"title": "회의", "action_items": ["보고서 검토"', runner="naive"),
    trial("js3", final_content="not json", runner="naive"),
]
specs = {
    "js1": SchemaSpec(required_keys=["title", "action_items"]),
    "js2": SchemaSpec(required_keys=["title", "action_items"]),
    "js3": SchemaSpec(required_keys=["title"]),
}
m6 = m6_schema_adherence.compute(trials_m6, specs)
print("M5", m5.value, "| M6 loose", m6.value, "strict", m6.breakdown.get("strict"), "repair_gain", m6.breakdown.get("repair_gain"))

**출력 해석:** M5(절제)와 M6(스키마 준수)를 함께 봅니다.

- `M5 0.5` — 절제 대상 2건 중 1건만 통과했습니다. `irr1`은 도구를 호출하지 않아 통과했고, `irr2`(naive)는 불필요한 `weather`를 호출해 실패했습니다.
- `M6 strict 0.333` → `loose 0.667` — strict(`JsonExtractor`)는 `js1`만 통과하지만, loose(+AutoRepair)는 **잘린 `js2`까지 복구**해 통과시킵니다.
- `repair_gain 0.333` = loose − strict — **하네스 repair 단계의 순 기여분**입니다. `js3`("not json")은 복구할 수 없어 strict와 loose 모두 실패합니다.

## Session 5. M7 Efficiency · M8 Redundancy · M9 Faithfulness

M7은 성공률 대비 토큰 비용, M8은 중복 호출률(낮을수록 좋음), M9는 근거 충실성을 봅니다. 토큰을 적게 쓰는 harness trial(e1·e3)과 토큰이 많고 중복 호출도 하는 naive trial(e2)을 섞어, 효율·중복·충실성이 어떻게 달라지는지 확인합니다. (M9 judge는 교육용 `LengthRatioJudge` 스텁입니다.)

### Session 5-1. 핵심 코드

**하는 일:** M7(효율)·M8(중복 호출률)·M9(충실성)을 harness/naive 혼합 trial로 계산합니다.

**정상:** `{"M7": ~5.22, "M8": 0.5, "M9": ~0.667}`

**의미:** naive의 토큰 증가는 M7을 낮추고, 중복 호출은 M8을 높입니다. grounding context에 얼마나 충실한지는 M9로 확인합니다.

In [ ]:
from eval.metrics import m7_efficiency, m8_redundancy, m9_faithfulness
from eval.metrics.m9_faithfulness import GroundingSpec, LengthRatioJudge

# (en) e2 (naive) bloats tokens and duplicates a search call; e1/e3 (harness) stay lean.
# (kr) e2(naive)는 토큰을 많이 쓰고 search를 중복 호출한다. e1/e3(harness)은 더 간결하다.
trials = [
    trial("e1", final_content="14일 환불", turns=2, input_tokens=80, output_tokens=20),
    trial("e2", final_content="hallucinated",
        tool_calls=[ToolCallRecord(name="search", arguments={"q": "a"}), ToolCallRecord(name="search", arguments={"q": "a"})],
        turns=5, input_tokens=300, output_tokens=100, runner="naive"),
    trial("e3", final_content="3~5 영업일 배송", turns=1, input_tokens=60, output_tokens=15),
]
# (en) Golds match the harness final_content so exact M1 (=TSR) is positive and M7 is meaningful.
# (kr) gold를 harness 결과와 일치시켜 exact M1(=TSR)이 양수가 되도록 하고, M7도 의미 있게 만든다.
golds = {"e1": TaskGold(task_id="e1", answer="14일 환불"), "e3": TaskGold(task_id="e3", answer="3~5 영업일 배송")}
m1_eff = m1_task_success.compute([trials[0], trials[2]], golds, mode="exact")
m7 = m7_efficiency.compute(trials, tsr=m1_eff.value)
m8 = m8_redundancy.compute(trials)
# (en) LengthRatioJudge is a deterministic TEST-ONLY stub (token overlap), not a production judge.
# (kr) LengthRatioJudge는 결정적 테스트 전용 스텁(토큰 겹침 기준)이며, 운영용 judge가 아니다.
m9 = m9_faithfulness.compute(
    trials,
    {"e1": GroundingSpec(context="환불 14일 이내"), "e2": GroundingSpec(context="no promo"), "e3": GroundingSpec(context="배송 3~5 영업일")},
    LengthRatioJudge(),
)
print(json.dumps({"M7": m7.value, "M8": m8.value, "M9": m9.value}, indent=2))

**출력 해석:** 합성 trial 기준으로 효율·중복·충실성을 함께 봅니다.

- `M7 ≈ 5.22` — TSR=1.0(e1·e3 정답 일치)을 1000토큰당 효율로 환산한 값입니다. naive `e2`의 토큰 사용량(300+100)이 평균 토큰 수를 끌어올려 효율을 낮춥니다.
- `M8 0.5` — 중복 호출률입니다(낮을수록 좋음). `e2`가 같은 `search`를 2번 호출했기 때문에 전체 호출 2건 중 1건이 중복입니다.
- `M9 ≈ 0.667` — 3건 중 grounding context에 충실한 비율입니다. 여기서는 교육용 `LengthRatioJudge` 스텁을 사용하므로 토큰 겹침으로 근사합니다.

## Session 6. M10 Empty-response Recovery

빈 응답(또는 reasoning-only 응답)이 나오면 하네스가 재시도로 복구합니다. M10은 이 복구가 얼마나 성공했는지 보는 지표입니다. naive runner는 복구 경로가 없어 0점입니다.


### Session 6-1. M10 Empty-response Recovery

**하는 일:** 빈 응답 트리거를 복구한 비율(M10)을 harness/naive trial로 계산합니다.

**정상:** `M10 0.5 | micro 0.5`

**의미:** 빈 응답(또는 reasoning-only 응답)을 하네스가 재시도로 복구하는지 봅니다. naive는 복구 경로가 없어 0점입니다.

In [ ]:
from eval.metrics import m10_empty_recovery

# (en) harness recovered 2/2 empty triggers; naive recovered 0/2 -> mean 0.5.
# (kr) harness는 빈 응답 2건 중 2건을 복구하고, naive는 2건 중 0건을 복구한다. 평균은 0.5다.
t_rec_h = trial("r1", runner="harness", metadata={"recovery": {"empty_triggers": 2, "recovery_successes": 2}})
t_rec_n = trial("r2", runner="naive", metadata={"recovery": {"empty_triggers": 2, "recovery_successes": 0}})
m10 = m10_empty_recovery.compute([t_rec_h, t_rec_n])
print("M10", m10.value, "| micro", m10.breakdown.get("micro_recovery_rate"))


**출력 해석:** M10(빈 응답 복구)을 봅니다.

- `M10 0.5` — harness는 빈 응답 트리거 2건을 모두 복구해 1.0이고, naive는 하나도 복구하지 못해 0.0입니다. 둘의 평균이 0.5입니다.
- `micro 0.5` — trial 가중이 아니라 **트리거 건수 가중** 복구율입니다. 전체 트리거 4건 중 2건을 복구했으므로 0.5이고, 복구 경로가 있는 하네스만 점수를 얻습니다.

## Session 7. 10개 지표 합성 smoke + 검증

모든 지표를 한 번에 계산하고, 핵심 값이 기대대로 나오는지 mini-check로 확인합니다.


### Session 7-1. 10개 지표 합성 smoke + mini-check

**하는 일:** 모든 지표를 한 번에 계산하고, 핵심 값을 assert로 검증합니다.

**정상:** `smoke_* … PASS` 6줄이 출력되고 `assert`가 통과합니다(오류 없음).

**의미:** 합성 trial의 기대값을 mini-check로 확인해, API 키 없이도 지표 로직의 회귀를 잡습니다.

In [ ]:
from eval.metrics.m1_task_success import TaskGold as _TG

t_ok = trial("s1", final_content='{"city": "seoul"}', final_structured={"city": "seoul"},
     tool_calls=[ToolCallRecord(name="rag__retrieve", arguments={"q": "Seoul"})], turns=2, input_tokens=100, output_tokens=40)
t_abs = trial("s2", tool_calls=[])
t_red = trial("s3", tool_calls=[ToolCallRecord(name="search", arguments={"q": "x"}), ToolCallRecord(name="search", arguments={"q": "x"})],
      turns=4, input_tokens=200, output_tokens=80)

smoke = {
"M1": m1_task_success.compute([t_ok], {"s1": _TG(task_id="s1", answer={"city": "seoul"})}).value,
"M3": m3_tool_selection.compute([t_ok], {"s1": [ToolCallRecord(name="retrieve", arguments={"q": "Seoul"})]}).value,
"M4": m4_argument_f1.compute([t_ok], {"s1": [ToolCallRecord(name="retrieve", arguments={"q": "Seoul"})]}).value,
"M5": m5_abstention.compute([t_abs], {"s2": True}).value,
"M6": m6_schema_adherence.compute([t_ok], {"s1": SchemaSpec(required_keys=["city"])}).value,
"M7": m7_efficiency.compute([t_ok, t_red], tsr=1.0).value,
"M8": m8_redundancy.compute([t_red]).value,
"M9": m9_faithfulness.compute([trial("s4", final_content="seoul population")], {"s4": GroundingSpec(context="seoul has many people")}, LengthRatioJudge()).value,
"M10": m10_empty_recovery.compute([t_rec_h, t_rec_n]).value,
}
_m2 = m2_pass_k.compute({"s1": [t_ok, t_ok]}, {"s1": _TG(task_id="s1", answer={"city": "seoul"})}, m1_task_success.score_trial_exact, ks=(1, 2))
smoke["M2_pass2"] = _m2.breakdown.get("pass_2", _m2.value)
print(json.dumps(smoke, indent=2))

checks = [
("smoke_M1", smoke["M1"] == 1.0),
("smoke_M6", smoke["M6"] == 1.0),
("smoke_M10", smoke["M10"] == 0.5),
("smoke_M2_pass2", smoke["M2_pass2"] == 1.0),
("datasets_non_empty", len(available_datasets()) >= 8),
("tau_bench_registered", "tau_bench.retail" in available_datasets()),
]
for name, ok in checks:
    print(name, "PASS" if ok else "FAIL")
assert all(ok for _, ok in checks), "eval mini-check failed"


**출력 해석:** 10개 지표를 한 번에 계산하고 핵심 값을 assert로 검증합니다.

- `smoke_M1/M6/M10/M2_pass2` 4개 **지표값**과 `datasets_non_empty`·`tau_bench_registered` 2개 **데이터셋** 검사가 모두 `PASS`입니다(API 키 없이 실행).
- 나머지 M3·M4·M5(=1.0)·M7(≈4.76)·M8·M9(=0.5)는 출력으로 확인합니다. smoke의 M7이 양수인 이유는 TSR=1.0인, 정답과 일치하는 trial을 사용했기 때문입니다(Session 5의 혼합 trial과 대비).
- `assert all(...)`가 통과하면 지표 로직 회귀 가드가 통과한 상태입니다.

## Session 8. (선택) `eval.run` — Cookbook matrix 스모크 테스트

API 키가 있으면 실제 데이터셋(BFCL+IFEval)으로 작은 매트릭스를 실행합니다. 산출물에는 상태와 경로만 남기고 원본 로그는 저장하지 않습니다.


### Session 8-1. (선택) eval.run Cookbook matrix 스모크 테스트

**하는 일:** API 키가 있으면 subprocess로 `eval.run`(BFCL+IFEval)을 실행해 실제 데이터셋 매트릭스를 스모크 테스트하고, 리포트 경로만 `run_note`에 남깁니다.

**정상:** `HAS_API=True` → `Saved: …/eval/reports/*.json|.md` + `returncode 0` / `HAS_API=False` → `[SKIP] …`

**의미:** 합성 지표가 실제 데이터셋에서도 실행되는지 최소 매트릭스로 확인합니다. 원본 로그는 저장하지 않습니다.

In [ ]:
import subprocess

run_note = None
if HAS_API:
    cmd = [sys.executable, "-m", "eval.run", "--dataset", "bfcl_v3.simple,ifeval",
           "--limit", "2", "--pass-k-trials", "1", "--runners", "harness"]
    print(" ".join(cmd))
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, timeout=900)
    saved = [ln.replace(str(ROOT), "<repo-root>") for ln in (proc.stdout or "").splitlines() if ln.startswith("Saved:")]
    run_note = {"returncode": proc.returncode, "status": "ok" if proc.returncode == 0 else "error",
                "report_paths": saved[-2:], "mode": "cookbook_matrix_smoke"}
    print(proc.stdout[-600:] if proc.stdout else proc.stderr[-400:])
else:
    # (en) No key -> skip the live matrix; the synthetic deliverables below still run.
    # (kr) API 키가 없으면 라이브 매트릭스를 건너뛴다. 아래 합성 산출물은 그대로 생성된다.
    run_note = {"status": "skipped", "reason": "no EXAONE_API_KEY", "mode": "cookbook_matrix_smoke"}
    print("[SKIP] eval.run needs EXAONE_API_KEY / EXAONE_BASE_URL")

**출력 해석:** API 키가 있을 때만 실제 데이터셋 매트릭스를 스모크 테스트합니다.

- `HAS_API=True`이면 `python -m eval.run …` 명령이 실행되고, `eval.run` 리포트 표의 마지막 부분(`stdout` 마지막 600자, 문장 중간부터 보일 수 있음)과 핵심 산출물인 `Saved: …/eval/reports/*.json|.md` **두 줄**이 보입니다(`returncode 0`). 라이브 호출이라 수 분 걸릴 수 있습니다.
- `HAS_API=False`이면 `[SKIP] …` 한 줄만 출력되고 `run_note.status="skipped"`로 기록됩니다. 나머지 산출물은 그대로 생성됩니다.

## Session 9. 산출물 — `metric_map.json` + `eval_table.md`


### Session 9-1. 산출물 저장 — metric_map.json · eval_table.md

**하는 일:** 지표 매핑·datasets·합성 smoke(+선택 run)를 `_out/`에 두 파일로 저장합니다.

**정상:** `saved: …/_out/metric_map.json`, `saved: …/_out/eval_table.md`

**의미:** 검토자와 CI가 읽기 좋은 JSON, 그리고 사람이 읽기 좋은 대시보드를 남깁니다.

In [ ]:
payload = {
"track": "track08", "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
"metric_map": [{"id": r[0], "name": r[1], "module": r[2], "notes": r[3]} for r in METRIC_MAP],
"datasets": available_datasets(),
"report_strategy": {
"cookbook_matrix": "bfcl_v3.simple+ifeval in one eval.run (Session 8, key-gated)",
"taubench_simulation": "tau_bench.retail,airline -> eval/reports/taubench",
},
"synthetic_smoke": smoke,
"run": run_note,
}
(out_dir / "metric_map.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

table_md = "\n".join(
["# Track 08 — M1–M10 synthetic dashboard", "", "| Metric | Synthetic value |", "|---|---|"]
+ [f"| {k} | {v} |" for k, v in smoke.items()]
+ ["", "Values from synthetic trials; see `eval/metrics/` for the scoring logic.", ""]
)
(out_dir / "eval_table.md").write_text(table_md, encoding="utf-8")
print("saved:", (out_dir / "metric_map.json").resolve())
print("saved:", (out_dir / "eval_table.md").resolve())

**출력 해석:** 두 산출물이 파일로 저장됩니다.

- `saved: …/_out/metric_map.json` — M1–M10 매핑, datasets, 합성 smoke 값, API 키가 있을 때의 `run` 요약을 담은 기계가 읽기 좋은 산출물입니다.
- `saved: …/_out/eval_table.md` — 같은 합성값을 사람이 읽기 좋은 대시보드 표로 저장합니다.
- `_out/`은 gitignore 대상이라 커밋되지 않습니다.

## Wrap-up. 마무리

이 노트북에서는 M1–M10 표준 지표를 합성 `TrialResult`로 **직접 계산**하고, mini-check로 한 번 더 검증한 뒤(API 키 없이), API 키가 있으면 `eval.run`으로 실제 데이터셋(BFCL+IFEval)에 연결했습니다.

**핵심 정리**

- **pass^k ≠ pass@k** — M2의 `pass_3=0.0`처럼 k회 *모두* 성공해야 통과합니다. 한 번만 실패해도 0입니다.
- **strict vs loose+repair** — M6의 `repair_gain 0.333`은 하네스 AutoRepair가 잘린 JSON을 복구해 얻은 순 기여분입니다.
- **점수뿐 아니라 원인** — M3의 `failures`, M7·M8의 토큰·중복, M10의 복구율처럼 *왜 그런 점수가 나왔는지*까지 남깁니다.

**지표 선택 가이드**

| Metric | 설명 / 목적 | 언제 쓰면 좋은가 | 같이 볼 것 |
|---|---|---|---|
| **M1 Task Success Rate** | 정답 또는 judge 판정 기준으로 과업 성공 여부를 측정합니다. | 골든셋 회귀, 모델·프롬프트 비교의 기본 성공률이 필요할 때. | M2, M7, M9 |
| **M2 pass^k Reliability** | 같은 과업을 k회 반복했을 때 모두 성공하는지로 재현 신뢰도를 봅니다. | 한 번의 성공보다 안정성이 중요한 운영 게이트나 τ-bench류 평가를 만들 때. | M1, 실패 분산 |
| **M3 Tool Selection Accuracy** | 필요한 도구 이름과 개수를 맞게 골랐는지 채점합니다. | tool router, function calling, MCP 도구 선택 오류를 찾을 때. | M4, M8 |
| **M4 Argument F1** | 도구 인자의 key-value 쌍이 정답과 얼마나 맞는지 봅니다. | 슬롯 추출, 날짜·금액·ID처럼 파라미터 정확도가 중요한 태스크를 볼 때. | M3, canonical args |
| **M5 Abstention Score** | 도구가 필요 없는 태스크에서 불필요한 호출을 참는지 봅니다. | irrelevance, no-answer, 안전한 거절, 근거 부족 요청을 평가할 때. | M9, hallucinated calls |
| **M6 Schema Adherence** | JSON/스키마 출력이 strict 및 loose+repair 경로에서 통과하는지 봅니다. | structured output, API payload, AutoRepair 효과를 검증할 때. | strict, loose, repair_gain |
| **M7 Trajectory Efficiency** | TSR 대비 턴 수와 토큰 사용량을 함께 보아 비용 효율을 봅니다. | 성공률이 비슷한 후보 중 더 짧고 저렴한 trajectory를 고를 때. | M1, mean_tokens, mean_turns |
| **M8 Redundancy Rate** | 같은 trial 안에서 중복된 도구 호출의 비율을 셉니다(낮을수록 좋음). | 루프, 과도한 재시도, 같은 검색·조회 반복을 잡을 때. | M3, M7, duplicate_calls |
| **M9 Faithfulness / Groundedness** | 최종 답변의 주장이 grounding context로 뒷받침되는지 평가합니다. | RAG, 검색 기반 답변, 출처 기반 QA의 환각 위험을 볼 때. | M1, M5, judge 품질 |
| **M10 Empty-response Recovery** | 빈 응답 또는 reasoning-only 응답을 하네스가 재시도로 복구한 비율입니다. | provider 응답 품질 문제, retry 경로, naive 대비 하네스 견고성을 볼 때. | recovery metadata, M6 |

**한계**

- 이 노트북의 값들은 **합성 trial** 기준입니다. 절대 성능이 아니라 *지표가 어떻게 동작하는지 보여주는 예시*입니다.
- M9 judge는 교육용 `LengthRatioJudge` 스텁입니다(운영용 judge가 아님). `eval.run`은 API 키가 있어야 실행됩니다.

## 체크포인트

- [ ] Session 7 mini-check의 6개 항목이 모두 PASS (M1=1.0, M6=1.0, M10=0.5, M2 pass^2=1.0 …) — **API 키 없이**.
- [ ] `metric_map.json`에 M1–M10 **10개** 매핑이 모두 기록 (`run`은 API 키가 없으면 `skipped`).
- [ ] `eval_table.md` 대시보드 생성.

**다음:** **08b — 내 골든셋**에서는 이 지표들을 팀 고정 테스트셋의 **회귀 게이트**로 묶습니다. 그다음에는 Track 09(프레임워크 브리지, 선택) 또는 Track 10 캡스톤으로 이어집니다.